# pKaMaP:
Determine RNA nucleotide pKa values from changes in DMS reactivity in response to pH.
This notebook demonstrates a full pKaMaP workflow.

In [ ]:
import pkamap
from pkamap.processing import process_json_files, generate_residue_dataframe
from pkamap.fitting import calculate_pka_values, add_structure_annotations
from pkamap.filtering import apply_quality_filters
from pkamap.comparison import summarize
from pkamap.plotting import plot_titration_curves

## 1. Configuration

Specify the desired sequences and experimental conditions... include path(s) to JSON file(s)

In [ ]:
CODES = ["C01HK"]

JSON_FILES = [
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run53.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run54.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run55.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run56.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run44.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run46.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/run48.json",
    "/home/yesselmanlab/jbstearns/pkamapgithub/data/azenta.json",
]

# Experimental conditions: (buffer_mM, temp, Mg_mM, color, label)
CONDITIONS = [
    (300, "room", 10, "blue",   "300mM room temp"),
    (300, 25,     10, "red",    "300mM 25C"),
    (50,  "room", 10, "orange", "50mM room temp"),
    (50,  25,     10, "green",  "50mM 25C"),
    (50,  25,     5,  "brown",  "50mM 25C 5mM Mg"),
    (50,  25,     0,  "purple", "50mM 25C 0mM Mg"),
]

## 2. Load & Process Sequencing Data

In [ ]:
df = process_json_files(JSON_FILES, CODES)
df_res = generate_residue_dataframe(df, min_aligned=2000)
print(f"Constructs: {df['name'].nunique()}")
print(f"Total Residues Evaluated: {len(df_res):,}")

## 3. Fit pKa Values & Annotate with Structure

In [ ]:
pka = calculate_pka_values(df_res)
pka_annotated = add_structure_annotations(pka, df)
print(f"Total fits: {len(pka_annotated):,}")

## 4. Quality Filtering

Remove pKa fits from residues that have an absent, nonlinear, or noisy response to pH (filtering parameters included in 'filtering.py' file)

In [ ]:
pka_annotated_filtered = apply_quality_filters(pka_annotated)

## 5. Summary

Compute inverse-variance-weighted pKa for each protonation site and condition.

In [ ]:
protonation_sites_from_data = (
    pka_annotated_filtered[["motif_sequence", "nt_-1", "nt_+0", "nt_+1"]]
    .drop_duplicates()
    .to_dict("records")
)

summary = summarize(pka_annotated_filtered, pka_annotated, CONDITIONS, protonation_sites_from_data)
summary

## 6. Plotting

Inspect individual Henderson-Hasselbalch fits and visually compare conditions.

Note: takes a long time to run if you don't filter the dataframe for desired motifs!

In [ ]:
plot_titration_curves(
    df_res, pka_annotated_filtered, CONDITIONS, pka_annotated,
)

## 7. Export

Export all generated DataFrames.

In [ ]:
df.to_csv("C01HK_construct_data.csv", index=False)
df_res.to_csv("C01HK_residue_data.csv", index=False)
pka_annotated.to_csv("C01HK_pka_annotated.csv", index=False)
pka_annotated_filtered.to_csv("C01HK_pka_annotated_filtered.csv", index=False)
summary.to_csv("C01HK_summary_pka.csv", index=False)

---
# C01HK: NMR Comparison

Processing for the 'pKa comparison library' (code=C01HK) which contains RNA structural motifs that have pH-dependent conformational change

## 8. Select Motifs of Interest (C01HK)

Filter for what motifs are to be evaluated

In [ ]:
from pkamap.nmr_reference_C01HK import (
    NMR_REFERENCE,
    MOTIFS_PKA,
    MOTIFS_PROTONATION,
    PROTONATION_SITES,
    PROTONATION_SITES_ALL,
    filter_to_protonation_sites,
    compare_to_nmr,
)

In [ ]:
# All library motifs (have protonation events observed by NMR)
motifs = pka_annotated_filtered[pka_annotated_filtered["motif_sequence"].isin(MOTIFS_PROTONATION)]

# Only motifs with residues that have known pKa values
pka_motifs = pka_annotated_filtered[pka_annotated_filtered["motif_sequence"].isin(MOTIFS_PKA)]

# Specific residues that have NMR-confirmed protonation events
residues = filter_to_protonation_sites(motifs, PROTONATION_SITES_ALL)

# Specific residues with NMR-confirmed protonation events that have measured pKa values
pka_residues = filter_to_protonation_sites(pka_motifs, PROTONATION_SITES)

print(f"Residues with published pKa values: {len(pka_residues)}")
print(f"All residues with confirmed protonation events: {len(residues)}")

## 9. NMR Summary & Comparison

Summarize pKa results for previously-known protonation sites and compare results to published NMR values.

In [ ]:
nmr_summary = summarize(pka_annotated_filtered, pka_annotated, CONDITIONS, PROTONATION_SITES_ALL)
nmr_summary

In [ ]:
comparison_stats = compare_to_nmr(
    summary_df=nmr_summary,
    nmr_ref_df=NMR_REFERENCE,
    conditions=CONDITIONS,
    plot_combined=True,
)
print(comparison_stats.to_string(index=False))

## 10. C01HK Export

In [ ]:
motifs.to_csv("C01HK_analysis_motifs.csv", index=False)
pka_motifs.to_csv("C01HK_analysis_pka_motifs.csv", index=False)
residues.to_csv("C01HK_analysis_residues.csv", index=False)
pka_residues.to_csv("C01HK_analysis_pka_residues.csv", index=False)
nmr_summary.to_csv("C01HK_analysis_nmr_summary_pka.csv", index=False)
comparison_stats.to_csv("C01HK_analysis_comparison_stats.csv", index=False)